In [4]:
# Ermöglicht das erneute Laden der .py Module, auch nach deren Modifikation ohne den Kernel neu starten zu müssen

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
import json
import sqlite3
import requests
import re
from html import unescape
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional
from urllib.parse import urlparse
from src.tagesschau_client import TagesschauClient
import os


True

In [12]:
#from tagesschau_client import TagesschauClient

from dotenv import load_dotenv
load_dotenv()

from src.tagesschau_client import TagesschauClient

client = TagesschauClient(
    api_config_path="config/api_config.json",
    regions_path="config/regions.json",
    source_regions_path="config/source_regions.json",
    url_region_keywords_path="config/url_region_keywords.json",
    filters_path="config/filters.json",
)

await client.collect_and_store()



🕒 Ingest watermark (from ingest_date): 2026-01-07T18:25:15

📊 TAGESSCHAU INGEST SUMMARY
🔹 Artikel von API (Index): 362
🕒 Nach Watermark relevant: 52
💾 Artikel gespeichert:     52
📄 Kein Fulltext:           0
❌ Fehlgeschlagen:          0
⏭️ Gefiltert (Typ):         113
⏭️ Gefiltert (Ressort):     0
⏭️ Gefiltert (Watermark):   197


In [28]:
# Dummy-Test for Qdrant Vector DB

from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from pathlib import Path
import random

# ----------------------------
# Config
# ----------------------------
QDRANT_PATH = Path("vector_store/qdrant_test_11")
COLLECTION = "test_collection"
DIM = 4  # mini vector dimension

# ----------------------------
# Init
# ----------------------------
QDRANT_PATH.mkdir(parents=True, exist_ok=True)
client = QdrantClient(path=str(QDRANT_PATH))

# ----------------------------
# Create collection
# ----------------------------
collections = [c.name for c in client.get_collections().collections]
if COLLECTION not in collections:
    print("Creating collection...")
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
    )

# ----------------------------
# Insert dummy points
# ----------------------------
points_to_insert = []

for i in range(10):
    vec = [random.random() for _ in range(DIM)]
    payload = {
        "name": f"doc_{i}",
        "category": "test" if i < 5 else "other",
    }

    points_to_insert.append(
        PointStruct(
            id=i,          # int ID
            vector=vec,
            payload=payload,
        )
    )

client.upsert(collection_name=COLLECTION, points=points_to_insert)
print("Inserted 10 dummy vectors.")

# ----------------------------
# Similarity search
# ----------------------------
query = [0.5, 0.5, 0.5, 0.5]

print("\nSearch results:")

res = client.query_points(
    collection_name=COLLECTION,
    query=query,
    limit=5,
)

for r in res.points:
    print(f"id={r.id} score={r.score:.4f} payload={r.payload}")

# ----------------------------
# Filtered search
# ----------------------------
from qdrant_client.models import Filter, FieldCondition, MatchValue

print("\nFiltered search (category=test):")

query_filter = Filter(
    must=[
        FieldCondition(
            key="category",
            match=MatchValue(value="test")
        )
    ]
)

res = client.query_points(
    collection_name=COLLECTION,
    query=query,
    limit=5,
    query_filter=query_filter,
)

for r in res.points:
    print(f"id={r.id} score={r.score:.4f} payload={r.payload}")

print("\nDone.")


Creating collection...
Inserted 10 dummy vectors.

Search results:
id=5 score=0.9869 payload={'name': 'doc_5', 'category': 'other'}
id=6 score=0.9644 payload={'name': 'doc_6', 'category': 'other'}
id=2 score=0.9444 payload={'name': 'doc_2', 'category': 'test'}
id=1 score=0.9366 payload={'name': 'doc_1', 'category': 'test'}
id=9 score=0.9065 payload={'name': 'doc_9', 'category': 'other'}

Filtered search (category=test):
id=2 score=0.9444 payload={'name': 'doc_2', 'category': 'test'}
id=1 score=0.9366 payload={'name': 'doc_1', 'category': 'test'}
id=0 score=0.8787 payload={'name': 'doc_0', 'category': 'test'}
id=3 score=0.8561 payload={'name': 'doc_3', 'category': 'test'}
id=4 score=0.7900 payload={'name': 'doc_4', 'category': 'test'}

Done.


In [30]:
DOCUMENTS = [
    {
        "id": 1,
        "title": "BMW Gewinn steigt",
        "text": "BMW hat im zweiten Quartal seinen Gewinn deutlich gesteigert. Der Absatz von Elektroautos wuchs stark."
    },
    {
        "id": 2,
        "title": "VW meldet Rekordzahlen",
        "text": "Volkswagen meldet einen starken Gewinn und Rekordumsätze durch steigende Nachfrage nach E-Fahrzeugen."
    },
    {
        "id": 3,
        "title": "FC Bayern gewinnt Spiel",
        "text": "Der FC Bayern München gewann gestern Abend mit 3:0 gegen Borussia Dortmund."
    },
    {
        "id": 4,
        "title": "Aktienmärkte unter Druck",
        "text": "Die Aktienmärkte stehen unter Druck, nachdem neue Inflationsdaten veröffentlicht wurden."
    },
    {
        "id": 5,
        "title": "Tesla mit Absatzproblemen",
        "text": "Tesla kämpft in Europa mit sinkenden Verkaufszahlen und wachsendem Wettbewerbsdruck."
    },
]


In [31]:
# semantic_search_demo.py

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from pathlib import Path

# ----------------------------
# Model
# ----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")

# ----------------------------
# Qdrant
# ----------------------------
QDRANT_PATH = Path("vector_store/qdrant_text_demo")
QDRANT_PATH.mkdir(parents=True, exist_ok=True)

client = QdrantClient(path=str(QDRANT_PATH))

COLLECTION = "news"
DIM = 384

if COLLECTION not in [c.name for c in client.get_collections().collections]:
    client.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
    )

# ----------------------------
# Insert documents
# ----------------------------
texts = [d["text"] for d in DOCUMENTS]
vectors = model.encode(texts)

points = []
for doc, vec in zip(DOCUMENTS, vectors):
    points.append(
        PointStruct(
            id=doc["id"],
            vector=vec.tolist(),
            payload={
                "title": doc["title"],
                "text": doc["text"],
            },
        )
    )

client.upsert(collection_name=COLLECTION, points=points)

print("✅ Documents indexed.")

# ----------------------------
# Query
# ----------------------------
query_text = "Welche Autohersteller haben gute Gewinne gemacht?"

query_vec = model.encode([query_text])[0]

res = client.query_points(
    collection_name=COLLECTION,
    query=query_vec.tolist(),
    limit=3,
)

print("\n🔍 Query:", query_text)
print("\n📄 Results:\n")

for r in res.points:
    print(f"Score: {r.score:.4f}")
    print("Title:", r.payload["title"])
    print("Text:", r.payload["text"])
    print("-" * 50)


/home/vscode/.local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Documents indexed.

🔍 Query: Welche Autohersteller haben gute Gewinne gemacht?

📄 Results:

Score: 0.4226
Title: BMW Gewinn steigt
Text: BMW hat im zweiten Quartal seinen Gewinn deutlich gesteigert. Der Absatz von Elektroautos wuchs stark.
--------------------------------------------------
Score: 0.3896
Title: VW meldet Rekordzahlen
Text: Volkswagen meldet einen starken Gewinn und Rekordumsätze durch steigende Nachfrage nach E-Fahrzeugen.
--------------------------------------------------
Score: 0.3422
Title: Tesla mit Absatzproblemen
Text: Tesla kämpft in Europa mit sinkenden Verkaufszahlen und wachsendem Wettbewerbsdruck.
--------------------------------------------------


In [36]:
import os
import asyncio
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PointStruct
from pathlib import Path
import libsql_client

# ----------------------------
# Config
# ----------------------------
QDRANT_PATH = Path("vector_store/qdrant_turso_test3")
COLLECTION = "turso_articles"
LIMIT = 5

# ----------------------------
# Env
# ----------------------------
TURSO_DB_URL = os.environ["TURSO_DB_URL"].replace("libsql://", "https://")
TURSO_AUTH_TOKEN = os.environ["TURSO_AUTH_TOKEN"]

# ----------------------------
# Embedding model
# ----------------------------
model = SentenceTransformer("all-MiniLM-L6-v2")
EMBED_DIM = 384

# ----------------------------
# Qdrant
# ----------------------------
QDRANT_PATH.mkdir(parents=True, exist_ok=True)
qdrant = QdrantClient(path=str(QDRANT_PATH))

if COLLECTION not in [c.name for c in qdrant.get_collections().collections]:
    qdrant.create_collection(
        collection_name=COLLECTION,
        vectors_config=VectorParams(size=EMBED_DIM, distance=Distance.COSINE),
    )

# ----------------------------
# Load 5 articles from Turso
# ----------------------------
async def main():
    print("🔌 Connecting to Turso...")
    db = libsql_client.create_client(
        url=TURSO_DB_URL,
        auth_token=TURSO_AUTH_TOKEN,
    )

    print("📥 Loading 5 articles from DB...")

    rs = await db.execute("""
    SELECT
        external_id,
        title,
        fulltext
    FROM articles
    WHERE
        title LIKE '%Auto%'
        OR title LIKE '%BMW%'
        OR title LIKE '%VW%'
        OR title LIKE '%Tesla%'
        OR title LIKE '%Wirtschaft%'
        OR title LIKE '%Industrie%'
    LIMIT 10
""")


    rows = rs.rows
    print(f"Loaded {len(rows)} articles.")

    # ----------------------------
    # Build embeddings
    # ----------------------------
    texts = []
    metadatas = []

    for r in rows:
        text = (r["title"] or "") + "\n\n" + (r["fulltext"] or "")
        texts.append(text)
        metadatas.append({
            "external_id": r["external_id"],
            "title": r["title"],
        })

    print("🧠 Computing embeddings...")
    vectors = model.encode(texts)

    # ----------------------------
    # Insert into Qdrant
    # ----------------------------
    points = []
    for i, (vec, meta) in enumerate(zip(vectors, metadatas)):
        points.append(
            PointStruct(
                id=i,
                vector=vec.tolist(),
                payload=meta,
            )
        )

    qdrant.upsert(collection_name=COLLECTION, points=points)

    print("✅ Inserted into Qdrant.")

    # ----------------------------
    # Query
    # ----------------------------
    query_text = "Was gibt es Neues zu Autoherstellern und Wirtschaft?"

    print("\n🔍 Query:", query_text)

    qvec = model.encode([query_text])[0]

    res = qdrant.query_points(
        collection_name=COLLECTION,
        query=qvec.tolist(),
        limit=5,
    )

    print("\n📄 Results:\n")

    for r in res.points:
        print(f"Score: {r.score:.4f}")
        print("external_id:", r.payload["external_id"])
        print("title:", r.payload["title"])
        print("-" * 60)

    await db.close()


if __name__ == "__main__":
    await main()


🔌 Connecting to Turso...
📥 Loading 5 articles from DB...
Loaded 10 articles.
🧠 Computing embeddings...
✅ Inserted into Qdrant.

🔍 Query: Was gibt es Neues zu Autoherstellern und Wirtschaft?

📄 Results:

Score: 0.3183
external_id: tagesschau_fm-story-swr-2c3825fd-4d8c-3e60-b189-39175f3808ca
title: Auto steht quer: Sperrung auf A81 nach Unfall bei Neuenstadt
------------------------------------------------------------
Score: 0.3122
external_id: c4f1fdef-8f2c-4cfe-86aa-7b2e1657f0e0
title: IHK zu Schwerin zieht negative Wirtschaftsbilanz für 2025
------------------------------------------------------------
Score: 0.3020
external_id: af069799-d332-434c-b892-220241325026
title: Landwirte wollen in MV erneut Autobahnauffahrten blockieren
------------------------------------------------------------
Score: 0.2985
external_id: tagesschau_fm-story-rbb_brandenburg-landwirte-proteste-autobahn-auffahrten-mercosur
title: Landwirte in Brandenburg kündigen Proteste an Autobahnen an
--------------------